# SemCor scan store with Qwen query embeddings

This notebook follows the same high-level flow as `hotpotqa_embeds_qwen.ipynb`:
it first builds a scan-only SemCor store without creating embeddings, then loads Qwen only after you choose a query term.
Each SemCor record keeps `synset_name` and `synset_definition`, and the final PCA scatter plot colors points by `synset_name`.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


In [1]:
from collections import defaultdict
from itertools import islice
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from transformers import AutoModel, AutoTokenizer

from prepare_semcor import iter_sentence_records, load_semcor_stats
from text_processing import normalize_text


In [2]:
BASE_DIR = Path("data/semcor")
NUM_SENTENCES = 30000

TEXT_ENCODER_NAME_OR_PATH = "/home/xiaoyue/Qwen3-Embedding-4B"
QWEN_BATCH_SIZE = 16
QWEN_MAX_LENGTH = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_PATH = Path(f"semcor_qwen_scan_store_{NUM_SENTENCES}.pkl")

stats = load_semcor_stats(BASE_DIR)
pd.Series(stats)


dataset_name                                                             SemCor
base_dir                                   /home/xiaoyue/LiteSemRAG/data/semcor
nltk_data_dir                 /home/xiaoyue/LiteSemRAG/data/semcor/raw/nltk_...
processed_dir                    /home/xiaoyue/LiteSemRAG/data/semcor/processed
resources                                            [semcor, wordnet, omw-1.4]
file_count                                                                  352
sentence_count                                                            37176
token_count                                                              820410
annotation_count                                                         778587
semantic_annotation_count                                                235079
noun_annotation_count                                                     88892
oov_entity_count                                                           9684
multiword_annotation_count              

In [3]:
qwen_tokenizer = None
qwen_model = None


class ConsoleProgressHandle:
    def update(self, text: str):
        print(text)


def create_progress_handle(initial_text: str):
    handle = display(initial_text, display_id=True)
    if hasattr(handle, "update"):
        return handle
    return ConsoleProgressHandle()


def atomic_pickle_dump(obj, path: Path):
    temp_path = path.with_suffix(path.suffix + ".tmp")
    try:
        with temp_path.open("wb") as handle:
            pickle.dump(obj, handle)
        temp_path.replace(path)
    finally:
        if temp_path.exists():
            temp_path.unlink()


def load_semcor_sentence_sample(base_dir: Path, num_sentences: int):
    return list(islice(iter_sentence_records(base_dir), num_sentences))


def build_token_char_spans(sentence_text: str, tokens: list[str]):
    spans = []
    cursor = 0

    for token in tokens:
        while cursor < len(sentence_text) and sentence_text[cursor].isspace():
            cursor += 1

        start_char = sentence_text.find(token, cursor)
        if start_char < 0:
            raise ValueError(
                f"Could not align token {token!r} inside sentence starting from char {cursor}.\n"
                f"text={sentence_text!r}"
            )

        gap_text = sentence_text[cursor:start_char]
        if any(not ch.isspace() for ch in gap_text):
            raise ValueError(
                f"Unexpected non-whitespace gap before token {token!r}.\n"
                f"gap={gap_text!r}\ntext={sentence_text!r}"
            )

        end_char = start_char + len(token)
        spans.append((start_char, end_char))
        cursor = end_char

    return spans


def collect_sentence_noun_annotations(sentence_record: dict):
    token_char_spans = build_token_char_spans(sentence_record["text"], sentence_record["tokens"])
    noun_records = []

    for annotation in sentence_record.get("noun_annotations", []):
        token_start = int(annotation["token_start"])
        token_end = int(annotation["token_end"])
        if token_end <= token_start:
            continue

        start_char = token_char_spans[token_start][0]
        end_char = token_char_spans[token_end - 1][1]
        surface_text = sentence_record["text"][start_char:end_char]
        normalized_surface = normalize_text(surface_text)
        normalized_tokens = [normalize_text(token) for token in annotation["tokens"]]

        noun_records.append(
            {
                "surface_text": surface_text,
                "normalized_surface": normalized_surface,
                "normalized_tokens": normalized_tokens,
                "char_span": (int(start_char), int(end_char)),
                "token_span": (token_start, token_end),
                **annotation,
            }
        )

    return noun_records


def build_semcor_scan_store(sentences):
    store = {
        "sentences": sentences,
        "index": defaultdict(list),
        "surface_index": defaultdict(list),
        "lemma_index": defaultdict(list),
        "synset_index": defaultdict(list),
        "config": {
            "num_sentences": len(sentences),
            "scan_only": True,
            "text_encoder_name_or_path": TEXT_ENCODER_NAME_OR_PATH,
            "embedding_model_family": "qwen",
            "prompt_method": PROMPT_METHOD,
            "output_path": str(OUTPUT_PATH),
        },
    }

    indexed_sentence_count = 0
    record_count = 0
    total_sentences = len(sentences)
    progress_handle = create_progress_handle(f"Scanned 0/{total_sentences} sentences")

    for sentence_index, sentence_record in enumerate(sentences):
        noun_records = collect_sentence_noun_annotations(sentence_record)
        if noun_records:
            indexed_sentence_count += 1

        for noun_record in noun_records:
            record = {
                "sentence_index": int(sentence_index),
                "sentence_id": sentence_record["sentence_id"],
                "sentence_text": sentence_record["text"],
                "annotation_index": int(noun_record["annotation_index"]),
                "surface_text": noun_record["surface_text"],
                "normalized_surface": noun_record["normalized_surface"],
                "tokens": list(noun_record["tokens"]),
                "normalized_tokens": list(noun_record["normalized_tokens"]),
                "token_span": tuple(noun_record["token_span"]),
                "char_span": tuple(noun_record["char_span"]),
                "lemma": noun_record["lemma"],
                "pos": noun_record["pos"],
                "sense_key": noun_record["sense_key"],
                "synset_name": noun_record["synset_name"],
                "synset_definition": noun_record["synset_definition"],
                "lexname": noun_record["lexname"],
            }

            store["surface_index"][record["normalized_surface"]].append(record)

            lemma_key = normalize_text(record["lemma"]) if record["lemma"] else ""
            if lemma_key:
                store["lemma_index"][lemma_key].append(record)

            synset_name = record.get("synset_name") or "(unknown synset)"
            store["synset_index"][synset_name].append(record)

            seen_query_tokens = set()
            for token in record["normalized_tokens"]:
                if token and token not in seen_query_tokens:
                    store["index"][token].append(record)
                    seen_query_tokens.add(token)

            record_count += 1

        if (sentence_index + 1) % 200 == 0 or sentence_index + 1 == total_sentences:
            progress_handle.update(f"Scanned {sentence_index + 1}/{total_sentences} sentences")

    store["stats"] = {
        "num_sampled_sentences": len(sentences),
        "num_indexed_sentences": indexed_sentence_count,
        "num_records": record_count,
        "num_unique_query_tokens": len(store["index"]),
        "num_unique_surface_terms": len(store["surface_index"]),
        "num_unique_lemmas": len(store["lemma_index"]),
        "num_unique_synsets": len(store["synset_index"]),
    }
    return store


def lookup_semcor_records(store, query_text, mode="token"):
    normalized_query = normalize_text(query_text.strip())
    if mode == "token":
        records = store["index"].get(normalized_query, [])
    elif mode == "surface":
        records = store["surface_index"].get(normalized_query, [])
    elif mode == "lemma":
        records = store["lemma_index"].get(normalized_query, [])
    elif mode == "synset":
        records = store["synset_index"].get(query_text, [])
    else:
        raise ValueError("mode must be one of: token, surface, lemma, synset")

    enriched_records = []
    for record in records:
        item = dict(record)
        item["lookup_query"] = normalized_query if mode != "synset" else query_text
        item["lookup_mode"] = mode
        enriched_records.append(item)
    return enriched_records


def lookup_semcor_dataframe(store, query_text, mode="token", limit=20):
    records = lookup_semcor_records(store, query_text, mode=mode)
    rows = []
    for record in records[:limit]:
        rows.append(
            {
                "sentence_index": record["sentence_index"],
                "sentence_id": record["sentence_id"],
                "surface_text": record["surface_text"],
                "tokens": record["tokens"],
                "lemma": record["lemma"],
                "synset_name": record["synset_name"],
                "synset_definition": record["synset_definition"],
                "sentence_text": record["sentence_text"],
            }
        )
    return pd.DataFrame(rows)


def token_frequency_dataframe(store, limit=50, return_dataframe=False):
    rows = [
        {
            "token": token,
            "num_records": len(records),
            "num_unique_synsets": len(
                {record.get("synset_name") or "(unknown synset)" for record in records}
            ),
        }
        for token, records in store["index"].items()
    ]
    df = pd.DataFrame(rows).sort_values(
        by=["num_records", "token"],
        ascending=[False, True],
        ignore_index=True,
    )
    if limit is not None:
        df = df.head(limit).copy()

    df.insert(0, "rank", range(1, len(df) + 1))

    if return_dataframe:
        return df

    print(df.to_string(index=False))


def load_qwen_encoder(name_or_path: str, device: str):
    loaded_tokenizer = AutoTokenizer.from_pretrained(
        name_or_path,
        local_files_only=True,
        trust_remote_code=True,
        use_fast=True,
    )

    if loaded_tokenizer.pad_token is None and loaded_tokenizer.eos_token is not None:
        loaded_tokenizer.pad_token = loaded_tokenizer.eos_token

    model_kwargs = {
        "local_files_only": True,
        "trust_remote_code": True,
    }
    if device == "cuda":
        model_kwargs["dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    loaded_model = AutoModel.from_pretrained(name_or_path, **model_kwargs)
    loaded_model.to(device)
    loaded_model.eval()
    return loaded_tokenizer, loaded_model


def ensure_qwen_loaded():
    global qwen_tokenizer, qwen_model
    if qwen_tokenizer is None or qwen_model is None:
        qwen_tokenizer, qwen_model = load_qwen_encoder(TEXT_ENCODER_NAME_OR_PATH, DEVICE)
        print(f"Loaded Qwen encoder on {DEVICE}: {TEXT_ENCODER_NAME_OR_PATH}")
    return qwen_tokenizer, qwen_model


def extract_sentence_context(sentence_text: str, span):
    start_char, end_char = span
    left_boundary = max(
        sentence_text.rfind(".", 0, start_char),
        sentence_text.rfind("!", 0, start_char),
        sentence_text.rfind("?", 0, start_char),
    )
    right_candidates = [
        sentence_text.find(".", end_char),
        sentence_text.find("!", end_char),
        sentence_text.find("?", end_char),
    ]
    right_candidates = [idx for idx in right_candidates if idx != -1]

    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = len(sentence_text) if not right_candidates else min(right_candidates) + 1
    context_raw = sentence_text[context_start:context_end]

    if not context_raw.strip():
        context_start = max(0, start_char - 120)
        context_end = min(len(sentence_text), end_char + 120)
        context_raw = sentence_text[context_start:context_end]

    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - context_start - left_trim
    local_end = end_char - context_start - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def build_qwen_prompt(query_text, context_text, local_span=None, prompt_method="word_context"):
    normalized_query = normalize_text(query_text.strip())

    if prompt_method == "word_context":
        return f"word: {normalized_query} context: {context_text.strip()}"

    if prompt_method == "target_marker":
        if local_span is None:
            raise ValueError("local_span is required when prompt_method='target_marker'")
        start_char, end_char = local_span
        return (
            f"{context_text[:start_char]}[TGT] {context_text[start_char:end_char]} [TGT]"
            f"{context_text[end_char:]}"
        )

    raise ValueError(f"Unsupported prompt_method: {prompt_method}")


def encode_qwen_batch(prompt_texts, tokenizer, model, device, max_length=512):
    encoded_inputs = tokenizer(
        prompt_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    encoded_inputs = {key: value.to(device) for key, value in encoded_inputs.items()}

    with torch.inference_mode():
        outputs = model(**encoded_inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.detach().to(torch.float32).cpu().numpy()


def collect_query_span_embeddings(
    store,
    query_text,
    mode="token",
    batch_size=QWEN_BATCH_SIZE,
    max_length=QWEN_MAX_LENGTH,
    prompt_method=PROMPT_METHOD,
):
    query_records = lookup_semcor_records(store, query_text, mode=mode)
    if not query_records:
        print(f"No records found for {query_text!r} with mode={mode!r}")
        return None

    tokenizer, model = ensure_qwen_loaded()
    normalized_query = normalize_text(query_text.strip()) if mode != "synset" else query_text

    progress_handle = create_progress_handle(
        f"Embedded 0/{len(query_records)} records for {normalized_query!r}"
    )
    embedded_records = []

    for batch_start in range(0, len(query_records), batch_size):
        batch_end = min(batch_start + batch_size, len(query_records))
        batch_records = query_records[batch_start:batch_end]

        prompt_texts = []
        context_texts = []
        local_spans = []

        for record in batch_records:
            context_info = extract_sentence_context(record["sentence_text"], record["char_span"])
            prompt_text = build_qwen_prompt(
                query_text,
                context_info["context_text"],
                local_span=context_info["local_span"],
                prompt_method=prompt_method,
            )

            prompt_texts.append(prompt_text)
            context_texts.append(context_info["context_text"])
            local_spans.append(tuple(context_info["local_span"]))

        batch_embeddings = encode_qwen_batch(
            prompt_texts,
            tokenizer,
            model,
            DEVICE,
            max_length=max_length,
        )

        for record, context_text, local_span, prompt_text, embedding in zip(
            batch_records,
            context_texts,
            local_spans,
            prompt_texts,
            batch_embeddings,
        ):
            item = dict(record)
            item["context_text"] = context_text
            item["local_span"] = local_span
            item["prompt_text"] = prompt_text
            item["prompt_method"] = prompt_method
            item["embedding"] = embedding
            embedded_records.append(item)

        progress_handle.update(
            f"Embedded {batch_end}/{len(query_records)} records for {normalized_query!r}"
        )

    return {
        "query_text": query_text,
        "normalized_query": normalized_query,
        "mode": mode,
        "record_count": len(embedded_records),
        "batch_size": batch_size,
        "max_length": max_length,
        "prompt_method": prompt_method,
        "device": DEVICE,
        "text_encoder_name_or_path": TEXT_ENCODER_NAME_OR_PATH,
        "records": embedded_records,
    }


def _print_grouped_sentence_matches(records):
    grouped_records = {}
    for record in records:
        synset_name = record.get("synset_name") or "(unknown synset)"
        synset_definition = record.get("synset_definition") or "(no definition available)"
        if synset_name not in grouped_records:
            grouped_records[synset_name] = {
                "definition": synset_definition,
                "sentences": [],
            }
        grouped_records[synset_name]["sentences"].append(record.get("sentence_text") or "")

    print("Sentence matches by synset:")
    for synset_name in sorted(grouped_records):
        group = grouped_records[synset_name]
        print(f"\n[{synset_name}]")
        print(f"definition: {group['definition']}")
        for idx, sentence_text in enumerate(group["sentences"], start=1):
            print(f"  {idx}. {sentence_text}")


def plot_query_span_embeddings_pca(query_result, annotate=False, return_data=False):
    if query_result is None:
        return None

    records = query_result["records"]
    if not records:
        print("No embeddings available for plotting.")
        return None

    if len(records) < 2:
        print(f"Only {len(records)} record found; at least 2 are required for PCA.")
        _print_grouped_sentence_matches(records)
        if return_data:
            return {"records": records}
        return None

    embeddings = np.stack([record["embedding"] for record in records]).astype(np.float32)
    coords = PCA(n_components=2).fit_transform(embeddings)

    synset_names = sorted({record.get("synset_name") or "(unknown synset)" for record in records})
    cmap = plt.cm.get_cmap("tab20", max(1, len(synset_names)))
    synset_to_color = {synset_name: cmap(idx) for idx, synset_name in enumerate(synset_names)}

    plt.figure(figsize=(9, 7))

    for synset_name in synset_names:
        indices = [
            idx
            for idx, record in enumerate(records)
            if (record.get("synset_name") or "(unknown synset)") == synset_name
        ]
        if not indices:
            continue
        plt.scatter(
            coords[indices, 0],
            coords[indices, 1],
            s=48,
            alpha=0.8,
            color=synset_to_color[synset_name],
            label=synset_name,
        )

    if annotate:
        for idx, record in enumerate(records):
            plt.annotate(
                f"{record['surface_text']} [{record['sentence_index']}]",
                (coords[idx, 0], coords[idx, 1]),
                fontsize=8,
                alpha=0.75,
            )

    plt.title(
        f"Qwen PCA for {query_result['normalized_query']!r} ({len(records)} records)"
    )
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend(title="synset_name", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    _print_grouped_sentence_matches(records)

    if not return_data:
        return None

    return {
        "records": records,
        "embeddings": embeddings,
        "coords": coords,
        "synset_to_color": synset_to_color,
    }


NameError: name 'PROMPT_METHOD' is not defined

In [ ]:
sampled_sentences = load_semcor_sentence_sample(BASE_DIR, NUM_SENTENCES)
print(f"Loaded {len(sampled_sentences)} SemCor sentences")
print(sampled_sentences[0]["sentence_id"])
print(sampled_sentences[0]["text"])
pd.DataFrame(collect_sentence_noun_annotations(sampled_sentences[0]))[
    [
        "annotation_index",
        "surface_text",
        "tokens",
        "char_span",
        "lemma",
        "synset_name",
        "synset_definition",
    ]
]


In [ ]:
if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open("rb") as handle:
        scan_store = pickle.load(handle)
    print(f"Loaded scan-only store from {OUTPUT_PATH}")
else:
    scan_store = build_semcor_scan_store(sampled_sentences)
    atomic_pickle_dump(scan_store, OUTPUT_PATH)
    print(f"Saved scan-only store to {OUTPUT_PATH}")

scan_store["stats"]


In [ ]:
token_frequency_dataframe(scan_store, limit=500)


In [ ]:
print(scan_store["config"])
lookup_semcor_dataframe(scan_store, "cell", mode="token", limit=20)


In [ ]:
query = "face"
query_mode = "token"
PROMPT_METHOD = "word_context"  # or "word_context" "target_marker"

query_embedding_result = collect_query_span_embeddings(
    scan_store,
    query,
    mode=query_mode,
    batch_size=QWEN_BATCH_SIZE,
    max_length=QWEN_MAX_LENGTH,
    prompt_method=PROMPT_METHOD,
)

pca_result = plot_query_span_embeddings_pca(
    query_embedding_result,
    annotate=False,
    return_data=True,
)
pca_result.keys() if pca_result is not None else None
